# Task 2 – Pessoa 3: validação temporal e protocolo final

Fronteiras temporais, teste final protegido, folds *expanding-window*, pré-processamento
ajustado dentro de cada fold e agregação dos resultados entre folds.
Depende de `predictors_task2.json`, gerado pelo notebook `task2.ipynb`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

HERE = Path.cwd()
DATA_PATH = HERE / "Charging_Data_educational.csv"
PREDICTORS_PATH = HERE / "predictors_task2.json"
PROTOCOL_PATH = HERE / "validation_protocol_task2.json"

NORMAL_END_CAUSES = {"Charging ends normally", "User stops charging"}
CATEGORICAL = ["charging_post_id", "location_information", "district_name", "tariff_period"]

# Primeiro dia do teste final; tudo o que começa antes é desenvolvimento.
TEST_START = pd.Timestamp("2021-08-01")
N_FOLDS = 4
FOLD_MONTHS = 3
# Razão de custos FN/FP = 4 (secção 6 do task2); provisório até à modelação.
THRESHOLD = 0.2

## 1. Dados, alvo e preditores elegíveis

In [2]:
df = pd.read_csv(DATA_PATH)
df.columns = (
    df.columns.str.strip().str.lower()
    .str.replace(r"[^a-zA-Z0-9]+", "_", regex=True).str.strip("_")
)
start_dt = pd.to_datetime(df["start_time"])
end_dt = pd.to_datetime(df["end_time"])
y = (~df["end_cause"].isin(NORMAL_END_CAUSES)).astype(int)

protocol_inputs = json.loads(PREDICTORS_PATH.read_text(encoding="utf-8"))
eligible = protocol_inputs["eligible_predictors"]
excluded = {c for cols in protocol_inputs["excluded_by_decision"].values() for c in cols}
assert not excluded.intersection(eligible), "Preditor excluído presente na lista elegível"

print(f"Sessões: {len(df):,} | período: {start_dt.min():%Y-%m-%d} a {start_dt.max():%Y-%m-%d}")
print(f"Prevalência global: {y.mean():.2%} | preditores elegíveis: {len(eligible)}")

Sessões: 441,077 | período: 2019-12-31 a 2021-12-31
Prevalência global: 18.58% | preditores elegíveis: 37


## 2. Fronteiras temporais e teste final protegido

O teste final (a partir de `TEST_START`, cerca de 30% das sessões, tal como o corte 70/30 do
`task1`) não entra em nenhum fold nem em nenhuma decisão de modelação. É avaliado uma única
vez, no fim, com o modelo e o limiar já fixados.

In [3]:
dev_mask = start_dt < TEST_START
test_mask = ~dev_mask
dev_idx = np.flatnonzero(dev_mask)
test_idx = np.flatnonzero(test_mask)

print(f"Desenvolvimento: {dev_mask.sum():,} sessões ({dev_mask.mean():.1%}), "
      f"{start_dt[dev_mask].min():%Y-%m-%d} a {start_dt[dev_mask].max():%Y-%m-%d}")
print(f"Teste final protegido: {test_mask.sum():,} sessões ({test_mask.mean():.1%}), "
      f"{start_dt[test_mask].min():%Y-%m-%d} a {start_dt[test_mask].max():%Y-%m-%d}")

Desenvolvimento: 309,930 sessões (70.3%), 2019-12-31 a 2021-07-31
Teste final protegido: 131,147 sessões (29.7%), 2021-08-01 a 2021-12-31


## 3. Folds expanding-window com purga

Cada fold treina com tudo o que terminou antes do início da validação e valida nos três meses
seguintes. A purga retira do treino as sessões que começaram antes da fronteira mas cujo
resultado ainda não era conhecido nessa data. As janelas de validação são contíguas e a
última termina em `TEST_START`; o primeiro fold tem menos treino (cerca de 7 meses).

In [4]:
val_starts = [TEST_START - pd.DateOffset(months=FOLD_MONTHS * (N_FOLDS - k)) for k in range(N_FOLDS)]
val_ends = val_starts[1:] + [TEST_START]

folds, fold_rows = [], []
for k, (v_start, v_end) in enumerate(zip(val_starts, val_ends), start=1):
    in_val = ((start_dt >= v_start) & (start_dt < v_end)).to_numpy()
    started_before = (start_dt < v_start).to_numpy()
    in_train = started_before & (end_dt < v_start).to_numpy()
    train_idx, val_idx = np.flatnonzero(in_train), np.flatnonzero(in_val)
    folds.append((train_idx, val_idx))
    fold_rows.append({
        "fold": k,
        "treino_inicio": start_dt.iloc[train_idx].min().date(),
        "treino_fim": v_start.date(),
        "validacao_fim": v_end.date(),
        "n_treino": len(train_idx),
        "n_validacao": len(val_idx),
        "purgadas": int((started_before & ~in_train).sum()),
        "prevalencia_treino": y.iloc[train_idx].mean(),
        "prevalencia_validacao": y.iloc[val_idx].mean(),
    })

fold_table = pd.DataFrame(fold_rows)
pd.set_option("display.width", 200)
fold_table.round(4)

,fold,treino_inicio,treino_fim,validacao_fim,n_treino,n_validacao,purgadas,prevalencia_treino,prevalencia_validacao
0,1,2019-12-31,2020-08-01,2020-11-01,79695,66045,36,0.1966,0.1868
1,2,2019-12-31,2020-11-01,2021-02-01,145748,57027,28,0.1921,0.1679
2,3,2019-12-31,2021-02-01,2021-05-01,202793,42189,10,0.1853,0.1532
3,4,2019-12-31,2021-05-01,2021-08-01,244950,64938,42,0.1798,0.1943


## 4. Verificações de consistência do esquema

In [5]:
for k, (tr, va) in enumerate(folds, start=1):
    assert np.intersect1d(tr, va).size == 0, f"fold {k}: treino e validação sobrepõem-se"
    assert end_dt.iloc[tr].max() < start_dt.iloc[va].min(), f"fold {k}: treino com informação posterior"
    assert np.intersect1d(np.concatenate([tr, va]), test_idx).size == 0, f"fold {k}: toca no teste final"
    assert len(tr) > 0 and len(va) > 0
assert start_dt[dev_mask].max() < start_dt[test_mask].min()
assert all(folds[i][1].max() < folds[i + 1][1].min() for i in range(N_FOLDS - 1))
assert val_ends[-1] == TEST_START
assert len(dev_idx) + len(test_idx) == len(df)
print("Esquema consistente: sem sobreposição, sem informação futura, teste final intocado.")

Esquema consistente: sem sobreposição, sem informação futura, teste final intocado.


## 5. Pré-processamento dentro de cada fold

As transformações sem estado (hora e dia da semana cíclicos) são calculadas uma vez.
Imputação, escala e `OneHotEncoder` fazem parte do `Pipeline`, que é clonado e ajustado só com
o treino de cada fold, para que nada da validação influencie as estatísticas aprendidas.

In [6]:
def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    X = frame[eligible].copy()
    hour = start_dt.dt.hour + start_dt.dt.minute / 60
    X["hour_sin"], X["hour_cos"] = np.sin(2 * np.pi * hour / 24), np.cos(2 * np.pi * hour / 24)
    X["weekday_sin"] = np.sin(2 * np.pi * start_dt.dt.weekday / 7)
    X["weekday_cos"] = np.cos(2 * np.pi * start_dt.dt.weekday / 7)
    X = X.drop(columns="start_time")
    X[CATEGORICAL] = X[CATEGORICAL].astype(str)
    numeric = [c for c in X.columns if c not in CATEGORICAL]
    X[numeric] = X[numeric].astype(float)
    return X


X = build_features(df)
numeric_cols = [c for c in X.columns if c not in CATEGORICAL]

preprocessing = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), CATEGORICAL),
])
# Modelo de referência só para exercitar o esquema; o modelo final é escolhido depois.
reference_model = Pipeline([
    ("prep", preprocessing),
    ("clf", LogisticRegression(max_iter=1000)),
])

## 6. Resultados por fold e baselines

Dois baselines por fold: a prevalência (PR-AUC de um classificador sem poder discriminativo)
e a taxa histórica de anomalias do posto nos últimos 30 dias, sem treino.

In [7]:
def fold_metrics(y_true, score, threshold=THRESHOLD):
    alert = score >= threshold
    tp = int((alert & (y_true == 1)).sum())
    fp = int((alert & (y_true == 0)).sum())
    fn = int((~alert & (y_true == 1)).sum())
    n = len(y_true)
    return {
        "pr_auc": average_precision_score(y_true, score),
        "roc_auc": roc_auc_score(y_true, score),
        "brier": brier_score_loss(y_true, score),
        "precision": tp / (tp + fp) if tp + fp else np.nan,
        "recall": tp / (tp + fn) if tp + fn else np.nan,
        "alertas_por_1000": 1000 * alert.sum() / n,
        "fp_por_1000": 1000 * fp / n,
        "fn_por_1000": 1000 * fn / n,
    }


history_score = df["post_previous_abnormal_rate_30d"].fillna(0).to_numpy()
rows, pooled_y, pooled_p = [], [], []
for k, (tr, va) in enumerate(folds, start=1):
    model = clone(reference_model).fit(X.iloc[tr], y.iloc[tr])
    p = model.predict_proba(X.iloc[va])[:, 1]
    y_va = y.iloc[va].to_numpy()
    rows.append({
        "fold": k, "n_val": len(va), **fold_metrics(y_va, p),
        "baseline_prevalencia_pr_auc": y_va.mean(),
        "baseline_historico_pr_auc": average_precision_score(y_va, history_score[va]),
    })
    pooled_y.append(y_va)
    pooled_p.append(p)

results = pd.DataFrame(rows)
results.round(4)

,fold,n_val,pr_auc,roc_auc,brier,precision,recall,alertas_por_1000,fp_por_1000,fn_por_1000,baseline_prevalencia_pr_auc,baseline_historico_pr_auc
0,1,66045,0.6159,0.8273,0.1070,0.4690,0.6656,265.0920,140.7525,62.4726,0.1868,0.4604
1,2,57027,0.5777,0.8153,0.1023,0.4339,0.6011,232.6267,131.6920,66.9858,0.1679,0.4387
2,3,42189,0.5010,0.8069,0.1011,0.3988,0.5867,225.3668,135.4856,63.3103,0.1532,0.3617
3,4,64938,0.6353,0.8342,0.1079,0.4869,0.6671,266.2848,136.6380,64.6925,0.1943,0.4688


## 7. Agregação entre folds

Métrica principal: média simples (cada período conta o mesmo) com desvio-padrão. A média
ponderada pelo tamanho da validação serve de verificação de sensibilidade, o pior fold mostra o
risco e a tendência (declive por fold) indica melhoria ou degradação ao longo do tempo. Com
apenas 4 folds, a tendência é indicativa e não um teste estatístico.

In [8]:
def aggregate(res: pd.DataFrame, metric: str, higher_is_better: bool = True) -> dict:
    v, w = res[metric], res["n_val"]
    return {
        "media": v.mean(),
        "desvio_padrao": v.std(ddof=1),
        "media_ponderada": np.average(v, weights=w),
        "pior_fold": v.min() if higher_is_better else v.max(),
        "tendencia_por_fold": np.polyfit(res["fold"], v, 1)[0],
    }


summary = pd.DataFrame({
    "pr_auc": aggregate(results, "pr_auc"),
    "roc_auc": aggregate(results, "roc_auc"),
    "brier": aggregate(results, "brier", higher_is_better=False),
    "recall": aggregate(results, "recall"),
    "precision": aggregate(results, "precision"),
    "alertas_por_1000": aggregate(results, "alertas_por_1000"),
    "baseline_prevalencia_pr_auc": aggregate(results, "baseline_prevalencia_pr_auc"),
    "baseline_historico_pr_auc": aggregate(results, "baseline_historico_pr_auc"),
}).T
summary.round(4)

,media,desvio_padrao,media_ponderada,pior_fold,tendencia_por_fold
pr_auc,0.5825,0.0594,0.5909,0.5010,-0.0019
roc_auc,0.8209,0.0122,0.8225,0.8069,0.0012
brier,0.1046,0.0034,0.1050,0.1079,0.0001
recall,0.6301,0.0422,0.6356,0.5867,-0.0010
precision,0.4472,0.0390,0.4525,0.3988,0.0018
alertas_por_1000,247.3425,21.3958,250.1053,225.3668,-0.3682
baseline_prevalencia_pr_auc,0.1756,0.0186,0.1781,0.1532,0.0008
baseline_historico_pr_auc,0.4324,0.0488,0.4393,0.3617,-0.0052


In [9]:
pooled_y, pooled_p = np.concatenate(pooled_y), np.concatenate(pooled_p)
print(f"PR-AUC pooled (só informativa; mistura modelos de folds diferentes): "
      f"{average_precision_score(pooled_y, pooled_p):.4f}")

PR-AUC pooled (só informativa; mistura modelos de folds diferentes): 0.5967


## 8. Protocolo exportado

In [10]:
protocol = {
    "target": "is_abnormal",
    "test_start": str(TEST_START.date()),
    "fold_scheme": "expanding-window com purga por end_time",
    "n_folds": N_FOLDS,
    "fold_months": FOLD_MONTHS,
    "folds": fold_table.astype({"treino_inicio": str, "treino_fim": str, "validacao_fim": str}).to_dict("records"),
    "primary_metric": "pr_auc",
    "threshold_provisional": THRESHOLD,
    "aggregation": ["media", "desvio_padrao", "media_ponderada", "pior_fold", "tendencia_por_fold"],
    "baselines": ["prevalencia (PR-AUC)", "post_previous_abnormal_rate_30d"],
    "n_eligible_predictors": len(eligible),
}
PROTOCOL_PATH.write_text(json.dumps(protocol, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Protocolo guardado em {PROTOCOL_PATH.name}")

Protocolo guardado em validation_protocol_task2.json


## 9. Protocolo final de avaliação

**Alvo e dados.** `is_abnormal` = `end_cause` fora de {*Charging ends normally*, *User stops charging*} (18,58% das sessões). Preditores: os 37 elegíveis do `task2` (inclui as 26 colunas de histórico); excluídos `end_cause`, `is_abnormal`, as colunas de fuga, `userid` e `order_creation_time`. A meteorologia é usada como previsão disponível no início da sessão, como declara o enunciado.

**Partição temporal.** Desenvolvimento: 2019-12-31 a 2021-07-31 (309 930 sessões, 70,3%). Teste final: 2021-08-01 a 2021-12-31 (131 147 sessões, 29,7%), protegido e avaliado uma só vez com modelo e limiar já fixados.

**Validação.** *Expanding-window* com 4 folds de 3 meses (validações: 2020-08 a 10, 2020-11 a 2021-01, 2021-02 a 04, 2021-05 a 07). O treino de cada fold usa só sessões que terminaram antes da validação (purga de 10 a 42 sessões por fold). O pré-processamento (imputação, escala, *one-hot*) é ajustado dentro do `Pipeline` com o treino de cada fold. O limiar é escolhido nos folds, nunca no teste.

**Métricas.** Principal: PR-AUC (classe positiva rara). Secundárias: ROC-AUC, Brier, precision e recall no limiar. Operacionais por 1000 sessões: alertas, falsos positivos e falsos negativos.

**Baselines.** Prevalência (PR-AUC igual à prevalência do fold, média 0,176) e taxa histórica de anomalias do posto nos últimos 30 dias (PR-AUC média 0,432). Qualquer modelo tem de superar o segundo, não só o primeiro.

**Agregação.** Média simples dos folds com desvio-padrão e pior fold; média ponderada pelo tamanho como verificação de sensibilidade; declive por fold para detetar degradação. A agregação *pooled* só serve para escolher o limiar, porque mistura modelos de folds diferentes.

**Resultado do modelo de referência** (regressão logística, limiar provisório 0,2): PR-AUC 0,583 ± 0,059 (pior fold 0,501), ROC-AUC 0,821 ± 0,012, recall 0,630, precision 0,447, cerca de 247 alertas por 1000 sessões. A média ponderada (0,591) é próxima da simples, e não há tendência clara de degradação (declive de PR-AUC de -0,002 por fold). A variação entre folds acompanha a prevalência da validação (0,153 a 0,194).

**Limitações.** Com 4 folds, o desvio-padrão e a tendência são indicativos. O limiar 0,2 gera um nível de alertas alto e é provisório até à modelação.

**Verificação de consistência.** Alvo, preditores e exclusões coincidem com `predictors_task2.json`. Os folds não se sobrepõem, não usam informação posterior à fronteira e não tocam no teste final (secção 4). A partição 70/30 coincide com a do `task1`. O protocolo exportado está em `validation_protocol_task2.json`.